## ▶️ Step 0 — set up the notebook (run this first!)

This notebook runs in **Google Colab**. The first cell installs what's needed,
downloads the camp toolbox, and connects the data from Google Drive.

**Before you run it**, make sure you've opened the shared camp Drive folder and
clicked **"Add shortcut to Drive"** (put the shortcut in *My Drive*) — that's how
the notebook finds the data file. Then run the cell below and click **Connect** on
the Drive pop-up. Wait for **✅ Setup complete**, then run the rest top to bottom.


In [ ]:
#@title ▶️ Run me first — set up the notebook  { display-mode: "form" }
# Press the ▶ button. (Double-click the title to see the code.)
import os, sys, glob

print("1/3  installing mne ...")
get_ipython().system('pip install -q "mne==1.10.1"')

print("2/3  downloading the camp toolbox ...")
get_ipython().system('wget -q -O camp_utils.py https://raw.githubusercontent.com/anarghya-das/decoding-the-brain-camp/main/camp_utils.py')

print("3/3  connecting Google Drive for the data ...")
from google.colab import drive
drive.mount("/content/drive")
hits = sorted(glob.glob("/content/drive/MyDrive/**/synapse_preprocessed.pkl", recursive=True))
assert hits, (
    "Could not find synapse_preprocessed.pkl in your Drive.\n"
    "Open the shared camp folder, click 'Add shortcut to Drive', put the shortcut "
    "in 'My Drive', then run this cell again."
)
os.environ["CAMP_DATA_PATH"] = hits[0]
os.environ["CAMP_OUTPUT_DIR"] = "/content/drive/MyDrive/DecodingBrain_outputs"
print(f"\n\u2705 Setup complete. Using data at: {hits[0]}")
print("Your figures will be saved to Drive > DecodingBrain_outputs.")


# Week 1 · Day 2 — Meet the Data

**Decoding the Brain: EEG Data Science for Sound Sensitivity Research**

Welcome to your first notebook! Yesterday you learned what hyperacusis is and
how the brain makes electricity we can record. Today you meet the actual data
from the **SYNAPSE study** — real brain recordings from real people.

### The big question of the whole camp
> Can brain signals tell apart people with sound sensitivity from healthy people?

### By the end of this notebook you will be able to
1. Load the SYNAPSE dataset
2. Explain what a "subject", a "task", a "channel", and an "epoch" are
3. Pull out one person's brain recording and look at its shape
4. Count how much data we have

**How these notebooks work:** Read the text, run the code cells (Shift+Enter),
and look for `# TODO` — that's where *you* write code. A green ✅ means you got
it; a red ❌ means try again. Don't worry about breaking anything.

## 1. Set up
We keep a small toolbox called `camp_utils` so you don't have to retype
boilerplate. We import it as `cu`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")   # hide harmless library warnings

import numpy as np
import matplotlib.pyplot as plt

import camp_utils as cu

print("Toolbox loaded. Tasks in this study:", cu.TASKS)

## 2. Load the data
The whole study is saved in one file. `load_camp_data()` opens it and prints a
summary. Run it and read the output carefully.

In [ ]:
data = cu.load_camp_data()

### What just happened?
`data` is a Python **dictionary** — a labeled container. The labels (keys) we
care about most:

| Key | What's inside |
|---|---|
| `exp_subjects` | IDs of the 18 people **with** sound sensitivity (the **EXP** group) |
| `ctrl_subjects` | IDs of the 10 **healthy controls** (the **CTRL** group) |
| `exp_epochs` | the EXP group's brain recordings, organized by task |
| `ctrl_epochs` | the CTRL group's brain recordings |
| `clinical_scores` | questionnaire scores (how severe each EXP person's symptoms are) |

Let's look at the subject lists.

In [ ]:
print("EXP group (sound-sensitive):", data["exp_subjects"])
print()
print("CTRL group (healthy):", data["ctrl_subjects"])

## 3. Vocabulary you need

- **Subject** = one person in the study (e.g. `EXP22`).
- **Task** = one of the 4 listening experiments each person did:
  - `pmt` Pupil Muscular Test (a baseline / control task)
  - `let` Listening Effort Test (understanding speech in noise)
  - `hlt` Hearing Loudness Test (sounds at 5 loudness levels)
  - `ast` Aversive Sound Test (annoying/triggering sounds)
- **Channel** = one electrode. We have 16 — eight behind each ear, worn like a
  sticker (this is "ear-EEG" using a device called **CEEGrid**).
- **Epoch** = one short clip of brain activity around a single sound. One person
  doing one task gives us many epochs (one per sound they heard).

Run this to see the friendly names:

In [ ]:
for task in cu.TASKS:
    print(f"  {task.upper()} — {cu.TASK_NAMES[task]}")

## 4. Grab one person's recording
`data["exp_epochs"]["let"]` is a **list**, one entry per EXP subject, in the
same order as `data["exp_subjects"]`. So entry `0` belongs to subject `0`.

Let's grab the Listening Effort recording for the very first EXP subject.

In [ ]:
first_subject = data["exp_subjects"][0]
epochs = data["exp_epochs"]["let"][0]

print("Subject:", first_subject)
print("This is an MNE 'Epochs' object:", type(epochs).__name__)
print(epochs)

## 5. Look inside the recording
An `Epochs` object knows a lot about itself. Run each line:

In [ ]:
print("Channel names:", epochs.ch_names)
print("Number of channels:", len(epochs.ch_names))
print("Number of epochs (sound clips):", len(epochs))
print("Sampling rate:", epochs.info["sfreq"], "Hz  (samples per second)")
print("Time of first sample:", epochs.times[0], "s")
print("Time of last sample:", epochs.times[-1], "s")

**Notice:** time starts at **−5 seconds** and ends at **+7 seconds**. The sound
plays at **time 0**. So we record 5 seconds *before* the sound (the quiet
"baseline") and several seconds *after*. That before/after comparison is the
heart of everything we'll do.

Also notice the channel names skip `03` and `06` — that's just how the hardware
is wired. **Always read `epochs.ch_names`; never assume all 16 are present**
(some subjects had a bad electrode removed).

## 6. The data as numbers
Under the hood, the recording is just a big grid of numbers. Pull it out with
`.get_data()`.

In [ ]:
signal = epochs.get_data()
print("Shape of the data:", signal.shape)
print("That means: (", signal.shape[0], "epochs,",
      signal.shape[1], "channels,", signal.shape[2], "time points )")

### ✏️ Your turn #1
Pull out the recording for the **first CTRL subject** doing the **`hlt`** task,
and print how many epochs it has.

*Hint:* copy the pattern from section 4, but use `ctrl_subjects` and
`ctrl_epochs["hlt"]`.

In [ ]:
# TODO: replace None with the right values
ctrl_subject = None        # the first CTRL subject's ID
ctrl_epochs = None         # their HLT recording

# --- check your work (don't edit below) ---
cu.check(ctrl_subject == data["ctrl_subjects"][0],
         f"Got subject {ctrl_subject}.",
         "ctrl_subject should be data['ctrl_subjects'][0].")
if ctrl_epochs is not None:
    print(f"{ctrl_subject} has {len(ctrl_epochs)} epochs in the HLT task.")

## 7. How much data do we have in total?
We'll loop over every subject and count epochs. `cu.iter_subjects()` is a handy
helper that loops over `(subject_id, epochs)` pairs and **skips missing ones**
for us.

In [ ]:
total_epochs = 0
for subject, ep in cu.iter_subjects(data, "exp", "let"):
    total_epochs += len(ep)
print("Total LET epochs across all EXP subjects:", total_epochs)

### ✏️ Your turn #2
Fill in the loop to count the **total number of AST epochs in the CTRL group**.

In [ ]:
ast_total = 0
# TODO: loop over CTRL subjects' "ast" recordings and add up len(ep)
for subject, ep in cu.iter_subjects(data, "ctrl", "ast"):
    pass  # replace this line

cu.check(ast_total > 0 and ast_total == sum(
    len(ep) for _, ep in cu.iter_subjects(data, "ctrl", "ast")),
    f"Counted {ast_total} AST epochs in CTRL. Nice.",
    "Did you add len(ep) to ast_total inside the loop?")

## 8. A peek at clinical scores
The EXP subjects filled out questionnaires. Higher score = worse symptoms.
We'll use these later to ask *"do worse symptoms show bigger brain differences?"*

In [ ]:
hq = cu.get_clinical_score(data, "EXP22", "HQ_Total")
print("EXP22's Hyperacusis Questionnaire score:", hq)
print("(The clinical cutoff for hyperacusis is", cu.CLINICAL_CUTOFFS["HQ_Total"], ")")

## 🎯 Wrap-up
You now know how to:
- load the data and read the summary,
- tell apart subjects / tasks / channels / epochs,
- pull out one recording and inspect its shape,
- loop over subjects safely with `cu.iter_subjects`.

**Discuss with a neighbor:** Why do you think we record 5 seconds of *quiet*
before every sound? What could we use it for?

➡️ **Next:** Notebook 02 — we'll actually *see* the brain's response to a sound.